# DatasetCollection population feature space

This notebook creates toy cell-level data for multiple datasets, then calculates dataset-by-population feature spaces with `DatasetCollection`. The toy data has ten cell types grouped into broader immune categories with sample-specific composition shifts.

In [ ]:
import numpy as np
import pandas as pd
from anndata import AnnData

import celldega as dega

In [ ]:
rng = np.random.default_rng(42)

cell_types = [
    "CD4 T",
    "CD8 T",
    "Treg",
    "B cell",
    "Plasma cell",
    "NK",
    "Monocyte",
    "Macrophage",
    "Dendritic cell",
    "Neutrophil",
]
immune_group = {
    "CD4 T": "T/NK",
    "CD8 T": "T/NK",
    "Treg": "T/NK",
    "NK": "T/NK",
    "B cell": "B/plasma",
    "Plasma cell": "B/plasma",
    "Monocyte": "myeloid",
    "Macrophage": "myeloid",
    "Dendritic cell": "myeloid",
    "Neutrophil": "granulocyte",
}

condition_templates = {
    "inflamed": np.array([5, 6, 2, 2, 1, 3, 5, 4, 3, 4], dtype=float),
    "cold": np.array([2, 2, 1, 4, 3, 1, 2, 2, 1, 1], dtype=float),
    "myeloid_rich": np.array([2, 2, 1, 1, 1, 1, 6, 7, 4, 2], dtype=float),
}

rows = []
for sample_idx in range(12):
    sample_id = f"sample_{sample_idx:02d}"
    patient_id = f"patient_{sample_idx // 2:02d}"
    condition = ["inflamed", "cold", "myeloid_rich"][sample_idx % 3]
    n_cells = int(rng.integers(500, 900))

    alpha = condition_templates[condition] * rng.uniform(0.8, 1.2, size=len(cell_types))
    proportions = rng.dirichlet(alpha)
    counts = rng.multinomial(n_cells, proportions)

    for cell_type, count in zip(cell_types, counts, strict=False):
        for _ in range(count):
            rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "condition": condition,
                    "cell_type": cell_type,
                    "immune_group": immune_group[cell_type],
                }
            )

obs = pd.DataFrame(rows)
obs.index = [f"cell_{i:05d}" for i in range(len(obs))]

adata = AnnData(
    X=np.ones((len(obs), 1)),
    obs=obs,
    var=pd.DataFrame(index=["dummy_gene"]),
)
adata

In [ ]:
dataset = dega.dataset.DatasetCollection(
    adata,
    dataset_col="sample_id",
    obs_columns=["patient_id", "condition"],
)

dataset.obs.head()

In [ ]:
cell_type_space = dataset.construct_population_space(
    category="cell_type",
    key="cell_type_population",
    output="percentage",
)

immune_group_space = dataset.construct_population_space(
    category="immune_group",
    key="immune_group_population",
    output="percentage",
)

list(dataset.mod)

In [ ]:
cell_type_space.to_df().head()

In [ ]:
immune_group_space.to_df().join(dataset.obs[["condition"]]).groupby("condition").mean()

Each modality has the same dataset/sample observation axis as `dataset.obs`, but a different feature axis. Here `cell_type_population` has ten variables and `immune_group_population` has four broader immune variables.